# Train our Acehnese → Indonesian model and publish the free web app
**Before you start:** menu **Runtime → Change runtime type → T4 GPU → Save**. Then **Runtime → Run all**.

1. The first cell asks for your **Hugging Face token** (free account at huggingface.co → Settings → Access Tokens →
   *Create new token* → type **Write** → copy). Paste it and press Enter. Press Enter with nothing to skip publishing.
2. Then it runs by itself for about 40 minutes: downloads the human-translated pairs (NusaX-MT, FLORES-200,
   Hikayat Abu Sammah), fine-tunes NLLB-200, keeps the best epoch on the validation set, and scores **base vs ours** on the test sets.
3. At the end it publishes **your model** and **your translator app** on Hugging Face (free) and prints the link.

In [ ]:
from getpass import getpass
HF_TOKEN = getpass('Hugging Face WRITE token (Enter = skip publishing): ').strip()
USE_QURAN = False        # optional extra training data (Acehnese Quran translation, (c) Tgk. Mahjiddin Jusuf): ask Andrie first

In [ ]:
import os
if not os.path.isdir('/content/aceh_code'):
    !git clone -q -b claude/dataset-product-build-108xu8 https://github.com/Mutaib-yye/aceh_code /content/aceh_code
%cd /content/aceh_code
!git pull -q
!pip install -q -r requirements.txt -U transformers sentencepiece huggingface_hub
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
# training data -> data/final/combined (leak-checked)
!python -m acehid fetch-nusax
if not os.path.exists('data/final/flores/dev.json'):
    !wget -q -O /content/flores200_dataset.tar.gz https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz && python -m acehid flores /content/flores200_dataset.tar.gz || echo 'FLORES download failed: training without it'
extra = ''
if USE_QURAN:
    !python -m acehid fetch-quran
    extra = '--quran'
!python -m acehid build-train {extra}
import json; m = json.load(open('data/final/combined/manifest.json'))
print('train', m['train']['n'], m['train']['by_source'], '| validation', m['validation']['n'], '| tests', {k: v['n'] for k, v in m['tests'].items()})

In [ ]:
# fine-tune (max 10 epochs, stops early). Out of GPU memory? add:  --batch-size 4
!python -m acehid train --epochs 10

In [ ]:
from IPython.display import Markdown
Markdown(open('models/ace-id-nllb/RESULTS.md', encoding='utf-8').read())

In [ ]:
# try it: change the sentences
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from acehid.normalize import normalize
tok = AutoTokenizer.from_pretrained('models/ace-id-nllb', src_lang='ace_Latn')
model = AutoModelForSeq2SeqLM.from_pretrained('models/ace-id-nllb').to('cuda').eval()
def tr(text):
    x = tok([normalize(text)[0]], return_tensors='pt').to('cuda')
    with torch.no_grad():
        y = model.generate(**x, forced_bos_token_id=tok.convert_tokens_to_ids('ind_Latn'), num_beams=4, max_new_tokens=128)
    return tok.batch_decode(y, skip_special_tokens=True)[0]
for s in ['Lon jak u peukan uroe nyoe.', 'Teuma troih le Abu Sammah, budak indah paraih rupa']:
    print(s, '->', tr(s))

In [ ]:
# publish: model -> huggingface.co/<you>/ace-id-nllb , app -> huggingface.co/spaces/<you>/penerjemah-aceh
import shutil
fp16 = 'models/ace-id-nllb-fp16'                       # half precision: 1.2 GB instead of 2.4 GB, same translations
model.half().save_pretrained(fp16); tok.save_pretrained(fp16)
for f in ('training_log.json', 'RESULTS.md', 'README.md'):
    shutil.copy(f'models/ace-id-nllb/{f}', fp16)
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    user = api.whoami()['name']
    model_repo, space_repo = f'{user}/ace-id-nllb', f'{user}/penerjemah-aceh'
    api.create_repo(model_repo, exist_ok=True)
    api.upload_folder(repo_id=model_repo, folder_path=fp16, commit_message='Fine-tuned Acehnese -> Indonesian NLLB')
    api.create_repo(space_repo, repo_type='space', space_sdk='gradio', exist_ok=True)
    api.add_space_variable(space_repo, 'MODEL_ID', model_repo)
    api.upload_folder(repo_id=space_repo, repo_type='space', folder_path='space', commit_message='Translator app', ignore_patterns=['__pycache__/*', '*.pyc'])
    print('MODEL :', f'https://huggingface.co/{model_repo}')
    print('APP   :', f'https://huggingface.co/spaces/{space_repo}', '(first start takes ~5 minutes)')
else:
    from google.colab import drive
    drive.mount('/content/drive'); shutil.copytree(fp16, '/content/drive/MyDrive/aceh_models/ace-id-nllb', dirs_exist_ok=True)
    print('No token: model saved to Google Drive > aceh_models > ace-id-nllb')